In [1]:
# -*- coding: utf-8 -*-
"""
Created on 2026-06-23

@author:       Oscar Trevizo
@institution:  Harvard Extension School — Graduate Data Science Program (2023)
@context:      Independent project — applying course concepts to real-world data
@environment:  Python 3.14.3 | myenv | MacBook Air M5

UN World Population Prospects — Data Product Vignette
=====================================================

Description:
    Demonstrates data product properties — metadata, semantic layer, quality
    reporting, and lineage tracking — using a single source: UN WPP 2024.

    The output is a country-year panel (1961–2023) exposed as a DataProduct
    with a human-readable card, a semantic layer for column aliasing, and a
    JSON contract sidecar written to output/.

    Pipeline:
        load_un_wpp() -> filter_countries() -> clean_types()

    Source: UN World Population Prospects 2024
    https://population.un.org/wpp/

    Support module: data_product_lib.py (same folder)

Key Concepts:
    Data Product     — self-describing, governed unit of data
    Metadata         — owner, source, license, version, refresh cadence
    Semantic Layer   — business-friendly aliases for raw column names
    Quality Report   — completeness %, row count, freshness
    Lineage Tracker  — append-only log of every pipeline transformation
    Contract JSON    — machine-readable sidecar combining all four above

Revision History:
    2026-06-23  Original development
                - load_un_wpp(), filter_countries(), clean_types() pipeline
                - DataProductMetadata, SemanticLayer, LineageTracker, DataProduct classes
                - JSON contract export and Parquet output
"""


'\nCreated on 2026-06-23\n\n@author:       Oscar Trevizo\n@institution:  Harvard Extension School — Graduate Data Science Program (2023)\n@context:      Independent project — applying course concepts to real-world data\n@environment:  Python 3.14.3 | myenv | MacBook Air M5\n\nUN World Population Prospects — Data Product Vignette\n=====================================================\n\nDescription:\n    Demonstrates data product properties — metadata, semantic layer, quality\n    reporting, and lineage tracking — using a single source: UN WPP 2024.\n\n    The output is a country-year panel (1961–2023) exposed as a DataProduct\n    with a human-readable card, a semantic layer for column aliasing, and a\n    JSON contract sidecar written to output/.\n\n    Pipeline:\n        load_un_wpp() -> filter_countries() -> clean_types()\n\n    Source: UN World Population Prospects 2024\n    https://population.un.org/wpp/\n\n    Support module: data_product_lib.py (same folder)\n\nKey Concepts:\n  

## What Is a Data Product?

A data product is a self-describing, governed unit of data that can be
discovered and consumed without tribal knowledge. Four properties distinguish
it from a plain DataFrame:

- **Discoverable** — carries metadata (owner, source, license, version) so
  consumers can find and evaluate it without asking the team.
- **Self-describing** — a schema and semantic layer map raw column names to
  business-friendly aliases with units and definitions.
- **Quality-governed** — a quality report (completeness, row count, freshness)
  is part of the artifact, not a separate spreadsheet.
- **Lineage-tracked** — every transformation step is logged so downstream
  consumers know exactly what was done to the data and in what order.

This vignette uses UN WPP as a single source to show each property. A second
vignette will add World Bank GDP and demonstrate why a shared semantic layer
makes a cross-source join trustworthy.

In [2]:
import numpy as np
import pandas as pd

from data_product_lib import LineageTracker

### Data Source

The UN World Population Prospects 2024 compact indicators file was downloaded
from [population.un.org/wpp](https://population.un.org/wpp/) on 2026-03-27
and stored locally at
`data/WPP2024_GEN_F01_DEMOGRAPHIC_INDICATORS_COMPACT_20260327.xlsx`.

The datestamped filename (`20260327`) reflects the publication date of that
release. To refresh, download the latest compact file from the WPP download
page and update `UN_FILE` in the cell below.

Fetching the file directly via API or URL is an exercise for a separate
vignette focused on data ingestion patterns.

## 1. Pipeline — Load → Filter → Clean

Three standalone functions replace the `WppPipeline.load_un()` / `merge()`
methods from the OO time-series notebook. Each step logs to a
`LineageTracker` so the transformations are auditable.

| Step | Function | What it does |
|---|---|---|
| 1 | `load_un_wpp()` | Read Estimates sheet, rename columns, year-filter, add migration direction |
| 2 | `filter_countries()` | Keep `LocType == 'Country'`; drop region/world aggregates |
| 3 | `clean_types()` | `pd.to_numeric` on seven numeric columns |

In [3]:
UN_FILE    = '../../data/WPP2024_GEN_F01_DEMOGRAPHIC_INDICATORS_COMPACT_20260327.xlsx'
YEAR_START = 1961
YEAR_END   = 2023


def load_un_wpp(filepath, year_start=YEAR_START, year_end=YEAR_END):
    """
    Load UN WPP Estimates sheet, rename columns, year-filter, and add
    migration-direction indicator columns.

    Parameters
    ----------
    filepath   : str, path to UN WPP Excel file.
    year_start : int, first year to include (inclusive). Default 1961.
    year_end   : int, last year to include (inclusive).  Default 2023.

    Returns
    -------
    pd.DataFrame — all location types (countries + regions + aggregates).
    """
    df = pd.read_excel(
        filepath,
        sheet_name='Estimates',
        skiprows=16,
        thousands=' ',
        usecols=[
            'Region, subregion, country or area *',
            'ISO3 Alpha-code',
            'ISO2 Alpha-code',
            'Type',
            'Year',
            'Total Population, as of 1 July (thousands)',
            'Median Age, as of 1 July (years)',
            'Population Growth Rate (percentage)',
            'Total Fertility Rate (live births per woman)',
            'Life Expectancy at Birth, both sexes (years)',
            'Net Number of Migrants (thousands)',
            'Net Migration Rate (per 1,000 population)',
        ]
    )

    df = df.rename(columns={
        'Region, subregion, country or area *'         : 'Location',
        'ISO3 Alpha-code'                              : 'ISO3',
        'ISO2 Alpha-code'                              : 'ISO2',
        'Type'                                         : 'LocType',
        'Total Population, as of 1 July (thousands)'  : 'Population_Ks',
        'Median Age, as of 1 July (years)'             : 'MedAge',
        'Population Growth Rate (percentage)'          : 'PopulationGrowthRate',
        'Total Fertility Rate (live births per woman)' : 'FertilityRate_births_per_woman',
        'Life Expectancy at Birth, both sexes (years)' : 'LifeExpectancy',
        'Net Number of Migrants (thousands)'           : 'NetMigrants_Ks',
        'Net Migration Rate (per 1,000 population)'    : 'NetMigrationRate_per_Kpop',
    })

    df = df.dropna(subset=['Year'])
    df['Year'] = df['Year'].astype(np.int64)
    df = df[(df['Year'] >= year_start) & (df['Year'] <= year_end)]

    # WPP uses 'Country/Area'; normalise to 'Country' for the filter step
    df.loc[df['LocType'] == 'Country/Area', 'LocType'] = 'Country'

    df['ReceivesMigrants']    = (df['NetMigrants_Ks'] > 0).astype(int)
    df['ImmigrantsEmigrants'] = df['NetMigrants_Ks'].apply(
        lambda x: 'Immigrants' if x > 0 else 'Emigrants'
    )

    return df.reset_index(drop=True)


def filter_countries(df):
    """
    Keep country-level rows only (LocType == 'Country').
    Drops region, sub-region, and world aggregate rows.
    """
    return df[df['LocType'] == 'Country'].reset_index(drop=True)


def clean_types(df):
    """
    Coerce seven demographic columns to float64.
    The UN file stores some values as strings (e.g. '...' for missing);
    errors='coerce' converts those to NaN.
    """
    numeric_cols = [
        'Population_Ks',
        'MedAge',
        'PopulationGrowthRate',
        'FertilityRate_births_per_woman',
        'LifeExpectancy',
        'NetMigrants_Ks',
        'NetMigrationRate_per_Kpop',
    ]
    df = df.copy()
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

In [4]:
lineage = LineageTracker()

# Step 1 — load
raw_df = load_un_wpp(UN_FILE)
lineage.log(
    step='1-load',
    operation='load_un_wpp',
    input_shape=(0, 0),
    output_shape=raw_df.shape,
    notes=f'Estimates sheet, skiprows=16, years {YEAR_START}–{YEAR_END}',
)

# Step 2 — filter
countries_df = filter_countries(raw_df)
lineage.log(
    step='2-filter',
    operation='filter_countries',
    input_shape=raw_df.shape,
    output_shape=countries_df.shape,
    notes='LocType == Country only',
)

# Step 3 — clean
df = clean_types(countries_df)
lineage.log(
    step='3-clean',
    operation='clean_types',
    input_shape=countries_df.shape,
    output_shape=df.shape,
    notes='pd.to_numeric on 7 numeric columns',
)

In [5]:
# ── Sanity check 1 — shape and dtypes ──────────────────────────────────────
print(f'Shape : {df.shape}   (rows, cols)')
print(f'Years : {df.Year.min()} – {df.Year.max()}')
print(f'Countries : {df.Location.nunique()}')
print()
print(df.dtypes)

Shape : (14931, 14)   (rows, cols)
Years : 1961 – 2023
Countries : 237

Location                           object
ISO3                               object
ISO2                               object
LocType                            object
Year                                int64
Population_Ks                     float64
MedAge                            float64
PopulationGrowthRate              float64
FertilityRate_births_per_woman    float64
LifeExpectancy                    float64
NetMigrants_Ks                    float64
NetMigrationRate_per_Kpop         float64
ReceivesMigrants                    int64
ImmigrantsEmigrants                object
dtype: object


In [6]:
# ── Sanity check 2 — first rows ────────────────────────────────────────────
df.head()

,Location,ISO3,ISO2,LocType,Year,Population_Ks,MedAge,PopulationGrowthRate,FertilityRate_births_per_woman,LifeExpectancy,NetMigrants_Ks,NetMigrationRate_per_Kpop,ReceivesMigrants,ImmigrantsEmigrants
0,Burundi,BDI,BI,Country,1961,2835.119,16.332,3.045,7.023,43.364,15.094,5.324,1,Immigrants
1,Burundi,BDI,BI,Country,1962,2907.943,16.300,2.041,7.038,43.492,-15.378,-5.288,0,Emigrants
2,Burundi,BDI,BI,Country,1963,2969.841,16.189,2.171,7.067,43.559,-11.502,-3.873,0,Emigrants
3,Burundi,BDI,BI,Country,1964,3055.650,16.213,3.507,7.090,43.601,29.639,9.700,1,Immigrants
4,Burundi,BDI,BI,Country,1965,3141.367,16.220,2.046,7.113,41.844,-13.947,-4.440,0,Emigrants


In [7]:
# ── Sanity check 3 — null counts (expect some in NetMigration columns) ─────
df.isnull().sum()

Location                           0
ISO3                               0
ISO2                              63
LocType                            0
Year                               0
Population_Ks                      0
MedAge                             0
PopulationGrowthRate               0
FertilityRate_births_per_woman     0
LifeExpectancy                     0
NetMigrants_Ks                     0
NetMigrationRate_per_Kpop          0
ReceivesMigrants                   0
ImmigrantsEmigrants                0
dtype: int64

In [8]:
# ── Sanity check 4 — lineage so far ────────────────────────────────────────
pd.DataFrame(lineage.to_list())

,step,operation,input_shape,output_shape,notes,timestamp
0,1-load,load_un_wpp,"(0, 0)","(18711, 14)","Estimates sheet, skiprows=16, years 1961–2023",2026-06-24T12:21:16.016751
1,2-filter,filter_countries,"(18711, 14)","(14931, 14)",LocType == Country only,2026-06-24T12:21:16.018096
2,3-clean,clean_types,"(14931, 14)","(14931, 14)",pd.to_numeric on 7 numeric columns,2026-06-24T12:21:16.022132


---
## 2. Data Product — Assemble

With the clean DataFrame in hand we:
1. Instantiate `DataProductMetadata` (who owns this, where it came from, what licence covers it).
2. Build a `SemanticLayer` with five canonical business names.
3. Wrap everything into a `DataProduct` — the three lineage steps logged above carry over automatically.

In [9]:
from datetime import datetime, timezone
from data_product_lib import DataProductMetadata, SemanticLayer, DataProduct

In [10]:
metadata = DataProductMetadata(
    name              = 'UN WPP Net Migration',
    description       = 'Country-year panel of UN WPP 2024 demographic indicators, 1961–2023',
    domain            = 'Demographics',
    owner             = 'Oscar Trevizo',
    source            = 'UN World Population Prospects 2024',
    source_url        = 'https://population.un.org/wpp/',
    license           = 'CC BY 3.0 IGO',
    version           = '1.0',
    refresh_frequency = 'Every 2 years (UN WPP release cycle)',
    created_at        = datetime.now(timezone.utc).isoformat(),
)

semantic = SemanticLayer()
semantic.register('net_migration_rate', 'NetMigrationRate_per_Kpop',      'per 1,000 population',
                  'Net migrants per 1,000 residents, UN WPP')
semantic.register('net_migrants',       'NetMigrants_Ks',                  'thousands of people',
                  'Net number of migrants')
semantic.register('population',         'Population_Ks',                   'thousands',
                  'Total population, as of 1 July')
semantic.register('fertility_rate',     'FertilityRate_births_per_woman',   'births per woman',
                  'Total fertility rate')
semantic.register('life_expectancy',    'LifeExpectancy',                  'years',
                  'Life expectancy at birth')

dp = DataProduct(df, metadata, semantic, lineage)
print(f'Assembled: {dp.metadata.name}')
print(f'  {len(dp.data):,} rows  |  {len(dp.semantic_layer.to_dict())} semantic entries  |  {len(dp.lineage.to_list())} lineage steps')

Assembled: UN WPP Net Migration
  14,931 rows  |  5 semantic entries  |  3 lineage steps


In [11]:
# ── Schema — inferred dtype per column ────────────────────────────────────
pd.DataFrame.from_dict(dp.schema(), orient='index', columns=['dtype']).rename_axis('column')

,dtype
column,
Location,object
ISO3,object
ISO2,object
LocType,object
Year,int64
Population_Ks,float64
MedAge,float64
PopulationGrowthRate,float64
FertilityRate_births_per_woman,float64


In [12]:
# ── Quality — completeness per column ─────────────────────────────────────
q = dp.quality()
print(f"Row count : {q['row_count']:,}")
print(f"Columns   : {q['column_count']}")
print(f"Freshness : {q['freshness']}")
print()
comp = pd.DataFrame.from_dict(q['completeness_pct'], orient='index', columns=['completeness_pct'])
comp[comp['completeness_pct'] < 100]

Row count : 14,931
Columns   : 14
Freshness : 2026-06-24T12:21:22.900100+00:00



,completeness_pct
ISO2,99.58


In [13]:
# ── Card — human-readable summary ─────────────────────────────────────────
dp.card()

  Data Product : UN WPP Net Migration
  Description  : Country-year panel of UN WPP 2024 demographic indicators, 1961–2023
  Domain       : Demographics
  Owner        : Oscar Trevizo
  Source       : UN World Population Prospects 2024
  License      : CC BY 3.0 IGO
  Version      : 1.0
  Refresh      : Every 2 years (UN WPP release cycle)
  Created      : 2026-06-24T12:21:22.900100+00:00
  Rows         : 14,931
  Columns      : 14
  Semantic layer (5 entries):
    net_migration_rate             → NetMigrationRate_per_Kpop  [per 1,000 population]
    net_migrants                   → NetMigrants_Ks  [thousands of people]
    population                     → Population_Ks  [thousands]
    fertility_rate                 → FertilityRate_births_per_woman  [births per woman]
    life_expectancy                → LifeExpectancy  [years]
  Incomplete columns (<100%):
    ISO2                                       99.6%
  Lineage (3 steps):
    [1-load] load_un_wpp  (0, 0) → (18711, 14)  // Esti

## 3. Semantic Layer — Query by Business Name

Instead of hard-coding `NetMigrationRate_per_Kpop`, consumers ask for
`net_migration_rate`. The semantic layer resolves the alias to the underlying
column and its unit — so downstream code does not break when the source
renames a column.

In [14]:
entry = dp.semantic_layer.get('net_migration_rate')
col   = entry['column']
unit  = entry['unit']

print(f"Business name : net_migration_rate")
print(f"Column        : {col}")
print(f"Unit          : {unit}")
print()

# Top-10 net receiving countries in 2020
top10 = (
    dp.data[['Location', 'Year', col]]
    .query('Year == 2020')
    .nlargest(10, col)
    .reset_index(drop=True)
)
top10.columns = ['Location', 'Year', f'net_migration_rate ({unit})']
top10

Business name : net_migration_rate
Column        : NetMigrationRate_per_Kpop
Unit          : per 1,000 population



,Location,Year,"net_migration_rate (per 1,000 population)"
0,"Bonaire, Sint Eustatius and Saba",2020,28.870
1,Saint Barthélemy,2020,26.653
2,Monaco,2020,26.255
3,Tokelau,2020,24.906
4,British Virgin Islands,2020,23.913
5,Maldives,2020,19.203
6,Turks and Caicos Islands,2020,15.365
7,Seychelles,2020,14.349
8,Gibraltar,2020,14.209
9,Slovenia,2020,13.562


## 4. Export

Two artifacts written to `output/`:
- **Parquet** — the data payload (typed, compressed, column-oriented).
- **Contract JSON** — schema + semantic layer + quality + lineage in one sidecar
  that travels with the data.

In [15]:
import os

out_dir = 'output'
os.makedirs(out_dir, exist_ok=True)

parquet_path  = f'{out_dir}/un_wpp_net_migration.parquet'
contract_path = f'{out_dir}/un_wpp_net_migration_contract.json'

dp.data.to_parquet(parquet_path, index=False)
print(f'Data written     : {parquet_path}  ({os.path.getsize(parquet_path):,} bytes)')

dp.contract(contract_path)

Data written     : output/un_wpp_net_migration.parquet  (569,973 bytes)
Contract written: output/un_wpp_net_migration_contract.json


## What's Next

This vignette wrapped a single source (UN WPP) as a governed data product.
The **next vignette** will add a second source — World Bank GDP indicators —
and wrap it the same way with its own `DataProductMetadata`, `SemanticLayer`,
and `LineageTracker`.

The shared semantic layer is what makes a cross-source join trustworthy:
instead of joining on raw column names that differ between sources, both
products expose canonical business names (`population`, `net_migration_rate`,
etc.). The join key is the agreed-upon name, not the implementation detail.

`data_product_lib.py` is already structured to support that second vignette
without modification.